# SAPTA Demo Notebook

**Production logic lives in `src/sapta/`.** This notebook is for storytelling and exploration.

Run from repo root with: `pip install -e .` then open this notebook.

## Problem

Rural pit latrines near wells create groundwater risk, especially after rainfall. Vacuum trucks have limited capacity—we need transparent prioritization and dispatch.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import seaborn as sns

from sapta.data.synthetic import generate_village_data
from sapta.scoring import score_dataframe
from sapta.dispatch import optimize_dispatch
from sapta.viz.maps import create_pitch_map

sns.set_theme(style="whitegrid")

## 1. Synthetic village data

30 pits near Northern Ghana (demo). See `data/README.md` for schema.

In [ ]:
df = generate_village_data()
df.head()

## 2. Priority scoring (SaniFlowCore)

Weighted heuristic index in [0, 1]: fill, rainfall (50 mm cap), distance decay (30 m), leach, vadose, condition.

In [ ]:
df = score_dataframe(df)
df[["pit_id", "fill_mm", "forecast_mm", "dist_to_well", "priority_score"]].head(10)

## 3. Greedy dispatch

Sort by score; assign DISPATCHED until truck capacity (10) or hours (48) at 2 h/pit.

In [ ]:
df = optimize_dispatch(df)
df[["pit_id", "priority_score", "status"]].head(15)

## 4. Map visualization

Red = high risk. Truck icon = dispatched. For a saved HTML map, run `python scripts/run_pipeline.py`.

In [ ]:
pitch_map = create_pitch_map(df)
pitch_map

## 5. Exploratory analysis

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["priority_score"], bins=15, color="#2E86AB")
plt.title("Distribution of Priority Scores")
plt.xlabel("Priority score")
plt.ylabel("Number of pits")
plt.tight_layout()
plt.show()

In [ ]:
corr = df["fill_mm"].corr(df["priority_score"])
print(f"Correlation fill_mm vs priority_score: {corr:.4f}")

plt.figure(figsize=(6, 4))
sns.scatterplot(data=df, x="fill_mm", y="priority_score", hue="status")
plt.title("Fill level vs priority score")
plt.tight_layout()
plt.show()

## 6. Rainfall sensitivity

How does forecast rainfall affect priority for a representative pit?

In [ ]:
import numpy as np
from sapta.scoring.engine import SaniFlowCore
from sapta.scoring.adapters import row_to_packets

engine = SaniFlowCore()
row = df.iloc[0]
telemetry, static_data = row_to_packets(row)

rain_range = np.linspace(0, 60, 25)
scores = []
for mm in rain_range:
    t = {**telemetry, "forecast_mm": mm}
    scores.append(engine.get_priority_index(t, static_data))

plt.figure(figsize=(7, 4))
plt.plot(rain_range, scores, marker="o")
plt.xlabel("Forecast rainfall (mm)")
plt.ylabel("Priority score")
plt.title(f"Rainfall sensitivity — {row['pit_id']}")
plt.tight_layout()
plt.show()

## Next steps

- CLI: `python scripts/run_pipeline.py`
- Tests: `pytest -q`
- Config: `configs/default.yaml`